# Tutorial 13-5: Visualizing Multi-Head Diversity
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. Why Multiple Heads?
If we only had one attention head, the model would be forced to 'average' all the important relationships in a sentence into a single set of weights. This would make it hard to focus on both the subject of a sentence and the object at the same time.

**The Multi-Head Advantage:**
* Each head has its own set of $W_q, W_k, W_v$ weights.
* Heads can specialize: one might focus on the next word, one on the previous word, and another on the verb related to a noun.
* The final output is a combination of all these diverse perspectives.

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertModel

# Load BERT
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, output_attentions=True)

sentence = "The scientist watched the stars through a powerful telescope."
inputs = tokenizer(sentence, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

with torch.no_grad():
    outputs = model(**inputs)
    # attentions is a tuple of 12 layers, each (batch, heads, seq, seq)
    attentions = outputs.attentions
    
# Let's look at Layer 11 (the last layer), which often has very specialized heads
last_layer_attn = attentions[10].squeeze(0) # Shape: (12, seq_len, seq_len)
print(f"Extracted {last_layer_attn.shape[0]} heads from the layer.")

## 2. Comparing Head Patterns
We will pick two different heads from the same layer and plot them. 

Observe how **Head 1** might have a 'diagonal' pattern (attending to the previous/next token), while **Head 5** might be 'vertical' (every word attending to a specific important word like the main verb).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 7))

head_idx_a = 0 # Look-next-word head
head_idx_b = 4 # Broad-context head

# Plot Head A
ax[0].imshow(last_layer_attn[head_idx_a].numpy(), cmap='Blues')
ax[0].set_title(f"Layer 11 - Head {head_idx_a + 1}\n(Note the specific diagonal focus)")
ax[0].set_xticks(range(len(tokens)))
ax[0].set_xticklabels(tokens, rotation=45)
ax[0].set_yticks(range(len(tokens)))
ax[0].set_yticklabels(tokens)

# Plot Head B
ax[1].imshow(last_layer_attn[head_idx_b].numpy(), cmap='Greens')
ax[1].set_title(f"Layer 11 - Head {head_idx_b + 1}\n(Note the more global/distributed focus)")
ax[1].set_xticks(range(len(tokens)))
ax[1].set_xticklabels(tokens, rotation=45)
ax[1].set_yticks(range(len(tokens)))
ax[1].set_yticklabels(tokens)

plt.tight_layout()
plt.show()

## 3. Interactive Visualization with BertViz
The `model_view` provides a bird's-eye view of all 12 heads across all 12 layers. You can see the 'lines of attention' change as you toggle heads on and off.

In [ ]:
%pip install bertviz

from bertviz import model_view

# This renders a large interactive window.
# Hover over the colors at the top to see specific heads!
model_view(attentions, tokens)

## 4. Discussion Point
Look at the `model_view` and try to find:
1. **The 'Delimiter' Head:** A head that only looks at the `[SEP]` or `[CLS]` tokens.
2. **The 'Syntax' Head:** A head that connects 'scientist' (Subject) to 'watched' (Verb).
3. **The 'Broad' Head:** A head where every word has a faint line to almost every other word (averaging context).